# 플래너 챗봇 실제 대화 테스트

이 노트북은 API 서버를 거치지 않고 `agents.todo_creation.planner.pipeline.run()`을 직접 호출합니다.

확인할 흐름:
- 플랜 생성과 무관한 대화는 `out_of_scope` 안내만 반환하고 플랜을 생성하지 않는지
- 목표 정보가 부족하면 헷갈리는 핵심만 꼬리 질문으로 묻는지
- 정보가 충분하면 TODO/캘린더 후보를 생성하는지

환경변수는 프로젝트 루트 `.env`를 읽습니다. `LLM_PROVIDER=runpod`이면 `RUNPOD_PLANNER_ENDPOINT_URL`을, 그 외에는 `QWEN_BASE_URL`/`QWEN_MODEL`을 사용합니다.

In [ ]:
import os
import sys
from datetime import date, datetime
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

from adapters.todo_creation.qwen_llm import DEFAULT_QWEN_MODEL, QwenLLM
from adapters.todo_creation.runpod_llm import RunPodQwenLLM
from agents.todo_creation.planner.pipeline import PlannerPorts, get_debug_state, run
from agents.todo_creation.schemas import PlannerInput


def build_real_planner_ports() -> PlannerPorts:
    provider = (os.getenv("LLM_PROVIDER", "qwen") or "qwen").strip().lower()
    if provider == "runpod":
        endpoint_url = os.getenv("RUNPOD_PLANNER_ENDPOINT_URL", "").strip()
        if not endpoint_url:
            raise RuntimeError("LLM_PROVIDER=runpod 이면 RUNPOD_PLANNER_ENDPOINT_URL 이 필요합니다.")
        llm = RunPodQwenLLM(
            endpoint_url=endpoint_url,
            api_key=os.getenv("RUNPOD_API_KEY", "EMPTY") or "EMPTY",
            adapter="planner",
            model=os.getenv("QWEN_MODEL", DEFAULT_QWEN_MODEL) or DEFAULT_QWEN_MODEL,
            max_tokens=int(os.getenv("PLANNER_MAX_TOKENS", "800")),
        )
        base_llm = RunPodQwenLLM(
            endpoint_url=endpoint_url,
            api_key=os.getenv("RUNPOD_API_KEY", "EMPTY") or "EMPTY",
            adapter="base",
            model=os.getenv("QWEN_MODEL", DEFAULT_QWEN_MODEL) or DEFAULT_QWEN_MODEL,
            max_tokens=int(os.getenv("PLANNER_MAX_TOKENS", "800")),
        )
        return PlannerPorts(llm=llm, classifier=base_llm, validator=base_llm)

    base_url = os.getenv("QWEN_BASE_URL", "").strip()
    model = os.getenv("QWEN_MODEL", "").strip() or DEFAULT_QWEN_MODEL
    if not base_url:
        raise RuntimeError("QWEN_BASE_URL 이 필요합니다. 예: http://localhost:11434/v1")
    llm = QwenLLM(
        base_url=base_url,
        model=model,
        api_key=os.getenv("QWEN_API_KEY", "EMPTY") or "EMPTY",
        max_tokens=int(os.getenv("PLANNER_MAX_TOKENS", "800")),
    )
    return PlannerPorts(llm=llm, classifier=llm, validator=llm)


ports = build_real_planner_ports()
thread_id = None
TODAY = date.today()
USER_ID = "notebook-user"

print("ROOT:", ROOT)
print("TODAY:", TODAY)
print("LLM:", type(ports.llm).__name__)
print("MODEL:", getattr(ports.llm, "model", None))

In [ ]:
def reset_chat() -> None:
    global thread_id
    thread_id = None
    print("thread reset")


def _dump_result(result):
    payload = result.model_dump(mode="json")
    print("kind:", payload.get("kind"))
    print("thread_id:", payload.get("thread_id"))
    if payload.get("kind") == "follow_up":
        print("question:", payload.get("question"))
        print("missing_aspects:", payload.get("missing_aspects"))
    elif payload.get("kind") == "out_of_scope":
        print("message:", payload.get("message"))
    else:
        print("summary_text:", payload.get("summary_text"))
        print("todos:")
        pprint(payload.get("todos"))
        print("calendar_events:")
        pprint(payload.get("calendar_events"))
    return payload


async def send(message: str):
    global thread_id
    print("USER:", message)
    result = await run(
        PlannerInput(
            user_id=USER_ID,
            message=message,
            today=TODAY,
            thread_id=thread_id,
        ),
        ports=ports,
        now=datetime.now(),
    )
    thread_id = result.thread_id
    return _dump_result(result)


def debug_state():
    if not thread_id:
        print("thread_id 없음")
        return None
    state = get_debug_state(thread_id=thread_id, ports=ports)
    pprint(state)
    return state

## 1. 무관 질문 차단

`kind`가 `out_of_scope`여야 하고, `todos`/`calendar_events`가 생성되면 안 됩니다.

In [ ]:
reset_chat()
await send("슈퍼스타 k 우승하고 싶은데 플랜을 어떻게 짜는게 좋을까?")

In [ ]:
await send("8월 8일에 경기가 예정되어 있어요")

In [ ]:
reset_chat()
await send("흑백요리사 우승하고 싶은데 어떻게 플랜을 짜는게 좋을까?")

## 2. 플랜 요청이지만 정보 부족

`kind`가 `follow_up`이어야 합니다. 부족한 정보 중 하나를 질문해야 정상입니다.

In [ ]:
reset_chat()
await send("정처기 공부 계획 짜줘")

## 3. 꼬리 질문에 이어서 답하기

위 셀에서 받은 질문에 맞춰 한 번씩 답해보세요. 같은 `thread_id`로 이어집니다.

예시 답변을 그대로 실행해도 되고, 문자열을 바꿔서 직접 대화해도 됩니다.

In [ ]:
await send("필기이고 시험은 3일 뒤야")

In [ ]:
await send("하루 2시간 가능하고 기출 1회독 했어. 비전공자야")

## 4. 한 번에 충분한 정보 제공

`kind`가 `candidates`이면 정상입니다. 오늘 날짜 task는 `todos`, 미래 날짜 task는 `calendar_events`에 들어갑니다.

In [ ]:
reset_chat()
await send("3일 뒤 정보처리기사 실기 시험이야. 하루 2시간 가능하고 SQL까지 봤고 전공자야. 남은 기간 공부 계획 짜줘")

## 5. 직접 대화용 셀

아래 문자열만 바꿔가며 실행하세요. 새 대화를 시작하려면 `reset_chat()`을 먼저 실행합니다.

In [ ]:
reset_chat()
await send("철인 삼종 경기에 출전하고 싶어요")

In [ ]:
debug_state()

## 6. 미지 목표·질문 횟수·말투 회귀 테스트

다음 시나리오에서 시험 질문이 나오지 않는지, 꼬리질문이 최대 두 번인지, 사용자 노출 문장에 `몽글`이 한 번만 포함되는지 확인합니다.

In [ ]:
reset_chat()
await send("흑백요리사 우승하고 싶어")
# 첫 질문에 답한 뒤 아래 두 줄을 순서대로 실행하세요.
# await send("아직 날짜는 정하지 않았고 가정 요리 경험은 있어요")
# await send("주 4회 가능하고 대표 메뉴를 완성하는 게 목표예요")
# debug_state()

In [ ]:
reset_chat()
chat_reply = await send("오늘 너무 피곤해")
visible_text = chat_reply.get("message", "")
print("몽글 사용 횟수:", visible_text.count("몽글"))

In [ ]:
reset_chat()
await send("8월 8일 철인 삼종 경기에 출전하고 싶어. 입문자이고 주 4회 훈련 가능해")
# 추가 질문에 답해 플랜이 생성되면 마지막 상세 일정이 TODAY+29 이내인지 확인하세요.
# long_state = debug_state()
# print(long_state)